In [2]:
# ============================================================
# CELL A — Imports & device
# ============================================================
import json, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
 
from torch_geometric.data import Batch
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, SAGPooling
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
 
# ── Same hyper-params used during training ──────────────────
HIDDEN_DIM = 64
HEADS      = 2
EMBED_DIM  = 64
GAT_BATCH  = 32

Device: cuda


In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00a 0:00:01


In [3]:
# ============================================================
# CELL B — Model definitions (must match training exactly)
# ============================================================
class GAT1(nn.Module):
    def __init__(self, input_dim=24, hidden_dim=HIDDEN_DIM, heads=HEADS):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.conv3 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.pool1 = SAGPooling(hidden_dim)
        self.pool2 = SAGPooling(hidden_dim)
        self.pool3 = SAGPooling(hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)
 
    def forward(self, data):
        x  = data.x.float()
        ei = data.edge_index
        ea = data.edge_attr.float() if data.edge_attr is not None else None
        b  = data.batch
        x  = self.fc1(x)
 
        x_r = x; x = self.conv1(x, ei, edge_attr=ea); x = self.bn1(x); x = F.relu(x + x_r)
        x, ei, ea, b, _, _ = self.pool1(x, ei, edge_attr=ea, batch=b)
 
        x_r = x; x = self.conv2(x, ei, edge_attr=ea); x = self.bn2(x); x = F.relu(x + x_r)
        x, ei, ea, b, _, _ = self.pool2(x, ei, edge_attr=ea, batch=b)
 
        x_r = x; x = self.conv3(x, ei, edge_attr=ea); x = self.bn3(x); x = F.relu(x + x_r)
        x, ei, ea, b, _, _ = self.pool3(x, ei, edge_attr=ea, batch=b)
 
        return global_mean_pool(x, b)
 
 
class GCN_Refiner(nn.Module):
    def __init__(self, in_dim=HIDDEN_DIM, out_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.gcn1 = GCNConv(out_dim, out_dim)
        self.gcn2 = GCNConv(out_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)
 
    def forward(self, X, ppi_edge_index):
        X = self.proj(X)
        X = self.bn1(F.relu(self.gcn1(X, ppi_edge_index)))
        X = self.bn2(F.relu(self.gcn2(X, ppi_edge_index)))
        return X
 
 
class ComplexPredictor(nn.Module):
    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64), nn.ReLU(),
            nn.Linear( 64,   1)
        )
 
    def forward(self, H, complex_indices_batch):
        logits = []
        for members in complex_indices_batch:
            idx    = torch.tensor(members, dtype=torch.long, device=H.device)
            pooled = H[idx].mean(0)
            logits.append(self.mlp(pooled))
        return torch.cat(logits, dim=0)

In [4]:
# ============================================================
# CELL C — Load data files (needed to build PPI graph + lookup)
# ============================================================
BASE = "/kaggle/input/datasets/shubhamkumar108/protein"   # adjust if path differs
 
with open(f"{BASE}/protein_index_map.json") as f:
    protein_index_map = json.load(f)
idx_to_pid = {v: k for k, v in protein_index_map.items()}
print(f"Protein index map : {len(protein_index_map)} proteins")
 
with open(f"{BASE}/proteinGraphsIndexed.pkl", "rb") as f:
    protein_graphs = pickle.load(f)
print(f"Protein graphs    : {len(protein_graphs)}")
 
ppi_df = pd.read_csv(f"{BASE}/positiveEdges_indexed.csv")
src, dst = ppi_df["Node1"].tolist(), ppi_df["Node2"].tolist()
ppi_edge_index = torch.tensor([src+dst, dst+src], dtype=torch.long).to(device)
print(f"PPI edges (bi)    : {ppi_edge_index.shape[1]}")

Protein index map : 7499 proteins
Protein graphs    : 7499
PPI edges (bi)    : 37528


In [7]:
# ============================================================
# CELL D — Instantiate models & load checkpoint
# ============================================================
gat_enc   = GAT1().to(device)
gcn_ref   = GCN_Refiner().to(device)
predictor = ComplexPredictor().to(device)
 
CKPT = "/kaggle/input/models/shubhamkumar108/proteincomplexpredictor/pytorch/complex-prediction/1/best_complex_model.pt"   # path to your saved .pt file
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
 
gat_enc.load_state_dict(ckpt["gat_enc"])
gcn_ref.load_state_dict(ckpt["gcn_ref"])
predictor.load_state_dict(ckpt["predictor"])
 
gat_enc.eval(); gcn_ref.eval(); predictor.eval()
print(f"Loaded checkpoint — epoch {ckpt['epoch']}, val AUC {ckpt['val_auc']:.4f}")

Loaded checkpoint — epoch 70, val AUC 0.9526


In [8]:
# ============================================================
# CELL E — Pre-compute embeddings once (reuse for all queries)
# ============================================================
@torch.no_grad()
def precompute_gat_embeddings():
    chunks  = [protein_graphs[i:i+GAT_BATCH]
               for i in range(0, len(protein_graphs), GAT_BATCH)]
    all_emb = []
    for chunk in chunks:
        pyg_batch = Batch.from_data_list(chunk).to(device)
        all_emb.append(gat_enc(pyg_batch).cpu())
    return torch.cat(all_emb, dim=0)   # [N, HIDDEN_DIM] on CPU
 
print("Pre-computing embeddings...")
raw_embs = precompute_gat_embeddings()   # do this ONCE, reuse below
with torch.no_grad():
    H_global = gcn_ref(raw_embs.to(device), ppi_edge_index).detach()
    # H_global shape: [7499, 64]  — one embedding per protein
print(f"H_global ready: {H_global.shape}")

Pre-computing embeddings...
H_global ready: torch.Size([7499, 64])


In [9]:
# ============================================================
# CELL F — predict_complex()
#
#  Accepts TWO input formats:
#    1. Integer indices:  predict_complex([7442, 2193, 2580])
#    2. UniProt IDs:      predict_complex(["P04637", "Q9BYX4", "P38398"])
#
#  You can mix:          predict_complex([7442, "Q9BYX4", 2580])
# ============================================================
def predict_complex(members, threshold=0.5):
    """
    members   : list of int indices OR UniProt ID strings (or mixed)
    threshold : float, default 0.5
    """
    # ── Resolve to integer indices ──────────────────────────
    resolved_idx   = []
    resolved_names = []
    errors         = []
 
    for m in members:
        if isinstance(m, int):
            # Already an integer index
            if m in idx_to_pid:
                resolved_idx.append(m)
                resolved_names.append(idx_to_pid[m])
            else:
                errors.append(f"Index {m} not in protein_index_map")
        else:
            # Treat as UniProt ID string
            uid = str(m).strip()
            if uid in protein_index_map:
                idx = protein_index_map[uid]
                resolved_idx.append(idx)
                resolved_names.append(uid)
            else:
                errors.append(f"UniProt ID '{uid}' not in protein_index_map")
 
    if errors:
        print("⚠️  Could not resolve some members:")
        for e in errors: print("   ", e)
        if len(resolved_idx) < 2:
            print("Need at least 2 valid members. Aborting.")
            return None
 
    # ── Score ───────────────────────────────────────────────
    with torch.no_grad():
        logit = predictor(H_global, [resolved_idx])
        prob  = torch.sigmoid(logit).item()
 
    label = "✅ COMPLEX" if prob >= threshold else "❌ NOT COMPLEX"
 
    print(f"\nInput   : {members}")
    print(f"Resolved: {resolved_names}  (indices: {resolved_idx})")
    print(f"Prob    : {prob:.4f}")
    print(f"Result  : {label}  (threshold={threshold})")
    return prob

In [16]:
# ============================================================
# CELL G — Try it out!
# ============================================================
 
# Using integer indices (from indexed_complexes.json — these are known positives)
predict_complex([7442, 2193, 2580])
predict_complex([8, 4, 3])
 
print("\n" + "─"*50)
 
# Using UniProt IDs — replace with any IDs from your protein_index_map.json
# first_three = list(protein_index_map.keys())[:3]
# predict_complex(first_three)
 
# A known negative (random proteins unlikely to form a complex)
predict_complex([0, 100, 200])
 
print("\n" + "─"*50)
 
# Batch prediction — score many complexes at once
def batch_predict(list_of_complexes, threshold=0.5):
    """
    list_of_complexes : list of lists, each inner list is one complex
    Returns           : list of (prob, label) tuples
    """
    results = []
    for cx in list_of_complexes:
        prob = predict_complex(cx, threshold=threshold)
        results.append(prob)
        print()
    return results
 
# Example batch:
# batch_predict([
#     [7442, 2193, 2580],   # known positive
#     [0, 100, 200],         # likely negative
#     ["P04637", "Q9BYX4"], # by UniProt ID
# ])


Input   : [7442, 2193, 2580]
Resolved: ['P25054', 'P30622', 'P46940']  (indices: [7442, 2193, 2580])
Prob    : 0.9829
Result  : ✅ COMPLEX  (threshold=0.5)

Input   : [8, 4, 3]
Resolved: ['A0A087WUM3', 'A0A087WY85', 'A0A0J9YYA3']  (indices: [8, 4, 3])
Prob    : 0.0077
Result  : ❌ NOT COMPLEX  (threshold=0.5)

──────────────────────────────────────────────────

Input   : [0, 100, 200]
Resolved: ['A0A0B4J1Y9', 'O00116', 'O00625']  (indices: [0, 100, 200])
Prob    : 0.0060
Result  : ❌ NOT COMPLEX  (threshold=0.5)

──────────────────────────────────────────────────
